# 04 — Backtesting cronológico

## Objetivo 

Este notebook implementa el protocolo de backtesting definido previamente para
comparar métodos tradicionales de Desarrollo y modelos de Machine Learning bajo
las mismas condiciones temporales.

El objetivo es reproducir una situación real de reservación:

1. se fija una fecha histórica de valuación;
2. únicamente se utiliza información disponible hasta dicha fecha;
3. se identifican las cohortes que aún no han alcanzado `dev_M`;
4. se genera una predicción de su Loss Incurred a madurez;
5. la predicción queda congelada;
6. posteriormente se revela el valor real observado en `dev_M`;
7. todas las metodologías se comparan sobre exactamente las mismas claves.

La característica fundamental del experimento es que el target futuro nunca
participa en la construcción de las predicciones.

A diferencia de una validación limitada al cierre 2025-12, la base simulada
contiene la trayectoria completa de desarrollo de cada cohorte. Esto permite
evaluar históricamente también observaciones con edades tempranas de desarrollo,
sin introducir información futura en los modelos.

## 1. Reglas del experimento

El backtesting se realiza independientemente para los tres escenarios simulados:

- `creciente`: `dev_M = 60`
- `decreciente`: `dev_M = 48`
- `mixto`: `dev_M = 48`

Las fechas históricas de valuación son mensuales desde `2020-01` hasta
`2025-11`.

Para cada combinación:

$$
(\text{scenario}, \text{valuation period}, \text{accident period})
$$

la información disponible queda determinada exclusivamente por:

$$
calendar\_period \leq valuation\_period
$$

Una cohorte es candidata a predicción cuando todavía no ha alcanzado `dev_M`
en la fecha de valuación.

El valor objetivo es:

$$
L_{a,M}
$$

donde:

- $a$ representa el período de ocurrencia;
- $M$ es el horizonte de madurez del escenario;
- $L_{a,M}$ es el Loss Incurred acumulado observado en `dev_M`.

### Separación entre entrenamiento y evaluación

Para una fecha de valuación $t$:

**Entrenamiento**

Solo pueden utilizarse cohortes cuyo `target_period` ya haya ocurrido:

$$
target\_period \leq t
$$

**Evaluación**

Se utilizan cohortes que todavía son inmaduras en $t$:

$$
target\_period > t
$$

El valor `target_amount` de estas cohortes se mantiene oculto durante todo el
proceso de predicción y únicamente se incorpora después de congelar los
resultados.

### Imports

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")

PROJECT_ROOT: c:\Users\derec\Desktop\Proyectos\Modelado_Reservas_ML_vs_metodos_Actuariales


In [4]:
from src.experiment.config import (
    DEV_M,
    VALUATION_PERIODS,
    FINAL_PERIOD,
    DEVELOPMENT_ALTERNATIVES,
)

from src.experiment.snapshots import (
    build_training_snapshots,
    build_train_test_snapshots,
)

## 2. Configuración experimental congelada

Las decisiones de diseño no se optimizan durante el backtesting.

Los horizontes de madurez, fechas de valuación, alternativas de Desarrollo,
features y configuraciones de modelos fueron definidos antes de observar los
resultados del backtesting.

Esto evita modificar el experimento en respuesta al desempeño observado.

In [5]:
DEV_M

{'creciente': 60, 'decreciente': 48, 'mixto': 48}

In [6]:
print(f"Número de valuaciones: {len(VALUATION_PERIODS)}")
print(f"Primera valuación:       {VALUATION_PERIODS[0]}")
print(f"Última valuación:        {VALUATION_PERIODS[-1]}")
print(f"Periodo final:           {FINAL_PERIOD}")

Número de valuaciones: 71
Primera valuación:       2020-01
Última valuación:        2025-11
Periodo final:           2025-12


In [7]:
print(f"Alternativas de Desarrollo: {len(DEVELOPMENT_ALTERNATIVES)}")

Alternativas de Desarrollo: 21


In [8]:
pd.DataFrame(DEVELOPMENT_ALTERNATIVES)

,0,1,2
0,S,24,none
1,S,24,minmax
2,S,24,std2
3,S,all,none
4,S,all,minmax
5,S,all,std2
6,VW,24,none
7,VW,24,minmax
8,VW,24,std2
9,VW,all,none


## 3. Snapshots históricos

Para cada fecha de valuación se construyen dos universos independientes.

### Snapshot de entrenamiento

Contiene exclusivamente cohortes cuyo Loss Incurred en `dev_M` ya sería
observable en esa fecha histórica.

Por tanto:

$$
target\_period \le valuation\_period
$$

Estas observaciones pueden utilizarse para estimar parámetros o entrenar modelos.

### Snapshot de evaluación

Contiene cohortes que todavía no han alcanzado `dev_M`.

Para ellas conocemos únicamente la trayectoria observada hasta la fecha de
valuación.

En esta etapa el target futuro se conserva fuera del dataset utilizado por los
modelos.

Esta separación representa la principal barrera contra leakage temporal.

In [9]:
def snapshot_summary(
    train: pd.DataFrame,
    test: pd.DataFrame,
    *,
    scenario: str,
    valuation_period,
) -> pd.DataFrame:
    
    return pd.DataFrame(
        {
            "scenario": [scenario],
            "valuation_period": [valuation_period],
            "dev_M": [DEV_M[scenario]],
            "n_train": [len(train)],
            "n_test": [len(test)],
            "test_age_min": [
                test["latest_dev_age"].min() if len(test) else np.nan
            ],
            "test_age_max": [
                test["latest_dev_age"].max() if len(test) else np.nan
            ],
        }
    )

### Valuación de prueba

In [10]:
scenario = "creciente"
valuation_period = pd.Period("2022-06", freq="M")

In [ ]:
df_scenario = simulations_base[scenario]

train_snapshot = build_training_snapshots(
    df=df_scenario,
    scenario=scenario,
    valuation_period=valuation_period,
    dev_M=DEV_M[scenario],
)

evaluation_snapshot = build_evaluation_snapshot(
    df=df_scenario,
    scenario=scenario,
    valuation_period=valuation_period,
    dev_M=DEV_M[scenario],
)